# Train results — R2

**Strategy R2: protected repair (reject/rollback on regression)**

This notebook reproduces the `R2` column of the **Train results** tab in `Results.xlsx`, end to end. It:

1. Runs the repository's own analyzer (`scripts/analyze_repair_strategies.py`) on the `R2` training run (`experiments/subset_train-r2`).
2. Reads the analyzer's `overall_summary.csv` and `mechanism_summary.csv`.
3. Derives the **common-core** metrics (shared by every strategy) and the **R2-specific** metrics.
4. Counts commonsense errors at the start and end of repair from the run's per-query result files.
   It also measures what happens after a rejection and what rollback costs.
5. Writes the exact table, plus its summary, to `Train_results_R2.xlsx`.

Everything is read-only analysis of an existing run: **no experiments, no model calls, no benchmark scoring.** The outputs below are produced by actually executing the cells.

## Step 0 — Setup and run the analyzer
We run from the repository root and point the analyzer at this strategy's training run. The analyzer writes its tables under a scratch output directory.

In [1]:
import subprocess, sys, os, csv, json, tempfile
from pathlib import Path

# Resolve the repository root (the folder that contains 'scripts/' and 'experiments/').
ROOT = Path.cwd()
while not (ROOT / 'scripts' / 'analyze_repair_strategies.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'scripts' / 'analyze_repair_strategies.py').exists(), 'Run this from inside the repo.'
NB_DIR = Path.cwd()  # folder containing this notebook (before we chdir to repo root)
os.chdir(ROOT)
PY = sys.executable  # the kernel's interpreter (the project venv)

STRATEGY = 'R2'
RUN_DIR = 'experiments/subset_train-r2'
OUT_DIR = str(Path(tempfile.gettempdir()) / 'train_nb_analysis' / STRATEGY)

assert Path(RUN_DIR).is_dir(), f'Missing run folder: {RUN_DIR}'
print('Repo root :', ROOT)
print('Strategy  :', STRATEGY)
print('Run folder:', RUN_DIR)

Repo root : C:\Users\Xue Jiarui\OneDrive - National University of Singapore\Documents\Agentic-Travel-Planner-Ablation-of-Verifications-
Strategy  : R2
Run folder: experiments/subset_train-r2


In [2]:
# Run the repo's analyzer for this one strategy. This only reads the saved run.
cmd = [PY, 'scripts/analyze_repair_strategies.py',
       '--run', f'{STRATEGY}={RUN_DIR}',
       '--output-dir', OUT_DIR, '--no-figures']
print('$', ' '.join(cmd))
res = subprocess.run(cmd, capture_output=True, text=True, encoding='utf-8',
                     env={**os.environ, 'PYTHONUTF8': '1'})
print(res.stdout.strip()[-1500:])
if res.returncode != 0:
    print('STDERR:', res.stderr[-2000:])
assert res.returncode == 0, 'analyzer failed'
print('\nAnalyzer exit code:', res.returncode)

$ C:\Users\Xue Jiarui\OneDrive - National University of Singapore\Documents\Agentic-Travel-Planner-Ablation-of-Verifications-\.venv\Scripts\python.exe scripts/analyze_repair_strategies.py --run R2=experiments/subset_train-r2 --output-dir C:\Users\XUEJIA~1\AppData\Local\Temp\train_nb_analysis\R2 --no-figures


R2: source=experiments\subset_train-r2, declared/evidenced queries=60, attempt queries=57, proposals=332, matrix cells=360

Analyzer exit code: 0


## Step 1 — Load the analyzer's summary tables
`overall_summary.csv` holds counts/means per run; `mechanism_summary.csv` holds the strategy-specific mechanism metrics.

In [3]:
def load_summary(path):
    d = {}
    with open(path, encoding='utf-8') as f:
        for row in csv.DictReader(f):
            d[row['metric']] = row
    return d

overall = load_summary(f'{OUT_DIR}/{STRATEGY}/overall_summary.csv')
mech = load_summary(f'{OUT_DIR}/{STRATEGY}/mechanism_summary.csv')
print('overall_summary metrics:', len(overall))
print('mechanism_summary metrics:', len(mech))
print('\nmechanism metrics available:')
for k, v in mech.items():
    print(f'  {k}: count={v["count"]} valid_n={v["valid_n"]} rate={v["rate"]}')

overall_summary metrics: 85
mechanism_summary metrics: 5

mechanism metrics available:
  rejected: count=25 valid_n=332 rate=0.0753012048192771
  harmful_candidate_blocked: count=25 valid_n=332 rate=0.0753012048192771
  fixed_target_but_protected_regression_rejected: count=0 valid_n=332 rate=0
  rejected_and_continued: count=20 valid_n=332 rate=0.0602409638554217
  rejected_plan_differs_from_accepted: count=25 valid_n=25 rate=1


## Step 2 — Common-core metrics (shared by every strategy)
These come straight from `overall_summary.csv`. 'Confirmed unchanged attempts' is shown as `N (pct%)` of total attempts, matching the R1 template already in the spreadsheet.

In [4]:
def iv(metric):
    return int(float(overall[metric]['value']))

queries              = iv('declared_queries')
repair_attempts      = iv('total_attempts')
personal_attempts    = iv('personal_attempts')
commonsense_attempts = iv('commonsense_attempts')
unchanged            = iv('unchanged_confirmed')
unchanged_pct        = round(unchanged / repair_attempts * 100, 2)
personal_fixes       = iv('personal_fixed_count_sum')
personal_regressions = iv('personal_newly_violated_count_sum')

core = [
    ('Queries', queries),
    ('Repair attempts', repair_attempts),
    ('Personal repair attempts', personal_attempts),
    ('Commonsense repair attempts', commonsense_attempts),
    ('Confirmed unchanged attempts', f'{unchanged} ({unchanged_pct}%)'),
    ('Personal constraint fixes', personal_fixes),
    ('Personal regressions', personal_regressions),
]
for k, v in core:
    print(f'  {k:40} {v}')

  Queries                                  60
  Repair attempts                          332
  Personal repair attempts                 141
  Commonsense repair attempts              191
  Confirmed unchanged attempts             235 (70.78%)
  Personal constraint fixes                2
  Personal regressions                     0


## Step 3 — R2-specific metrics
These describe R2's own mechanism, read from `mechanism_summary.csv`. Each is formatted `count / valid_n (rate%)`, with the rate to at most 2 decimals.

In [5]:
SPECIFIC = [('Harmful candidates blocked (rejected)', 'harmful_candidate_blocked', 'count_rate'), ('Rejected: fixed target but caused a protected regression', 'fixed_target_but_protected_regression_rejected', 'count_rate'), ('Rejected and loop continued', 'rejected_and_continued', 'count_rate'), ('Rejected candidate differed from accepted plan', 'rejected_plan_differs_from_accepted', 'count_rate')]

def fmt_pct(value):
    return f'{value:.2f}'.rstrip('0').rstrip('.')

def fmt_count_rate(row):
    c = int(float(row['count']))
    n = int(float(row['valid_n']))
    r = float(row['rate']) * 100
    return f'{c} / {n} ({fmt_pct(r)}%)'

specific = []
for label, metric, kind in SPECIFIC:
    if metric not in mech:
        specific.append((label, 'n/a (metric not produced for this run)'))
        continue
    row = mech[metric]
    specific.append((label, fmt_count_rate(row) if kind == 'count_rate' else row['rate']))
for k, v in specific:
    print(f'  {k:60} {v}')

  Harmful candidates blocked (rejected)                        25 / 332 (7.53%)
  Rejected: fixed target but caused a protected regression     0 / 332 (0%)
  Rejected and loop continued                                  20 / 332 (6.02%)
  Rejected candidate differed from accepted plan               25 / 25 (100%)


## Step 3b — Commonsense errors (start → end)
The analyzer does not report this, so it is counted directly from each query's saved result (`queries/<uid>/R2-*.json`): the commonsense errors in `initial_verification` (before any repair) versus `final_verification` (the plan R2 ended with). All 60 queries are included; queries with no repair attempts have identical start and end counts.

In [6]:
result_files = sorted(Path(RUN_DIR, 'queries').glob(f'*/{STRATEGY}-*.json'))
assert len(result_files) == queries, f'expected {queries} result files, found {len(result_files)}'

cs_start = cs_end = 0
for path in result_files:
    with open(path, encoding='utf-8') as f:
        result = json.load(f)
    cs_start += len(result['initial_verification']['commonsense_errors'])
    cs_end += len(result['final_verification']['commonsense_errors'])

cs_change = (cs_end - cs_start) / cs_start * 100
cs_row = ('Commonsense errors (start \u2192 end)',
          f'{cs_start} \u2192 {cs_end} ({cs_change:+.1f}%)'.replace('-', '\u2212'))
specific.append(cs_row)
print(f'  {cs_row[0]:60} {cs_row[1]}')

  Commonsense errors (start → end)                             304 → 271 (−10.9%)


## Step 3c — After a rejection, and what rollback costs
Also not produced by the analyzer, so these are read from the run's own records.

- **Queries never accepted again after their first rejection**: for each query with a rejection, every later attempt was also rejected (`attempts.jsonl`).
- **Rejected candidates identical to an earlier rejected one**: the model proposed exactly the same rejected plan again for the same query.
- **Tokens / time on rejected candidates**: the per-attempt `total_tokens_delta` and `attempt_time_seconds` of rejected attempts, out of all attempts. These per-attempt sums are slightly higher than the per-query totals in `summaries.csv`.
- **Commonsense fixes discarded by rejections**: each query is replayed from its initial plan using `repair_history` in `queries/<uid>/R2-*.json`. For a rejected attempt, an error counts as discarded when it is in the kept plan but not in the rejected candidate (errors are matched by their exact message). The replay must end at each query's saved final errors, which checks it.

In [7]:
from collections import Counter, defaultdict

with open(Path(RUN_DIR, 'attempts.jsonl'), encoding='utf-8') as f:
    attempts = [json.loads(line) for line in f if line.strip()]
by_query = defaultdict(list)
for a in attempts:
    by_query[a['query_uid']].append(a)
for steps in by_query.values():
    steps.sort(key=lambda a: a['step'])

# Queries whose first rejection was never followed by an accepted attempt.
rejected_queries = never_recovered = 0
repeat_rejections = 0
for steps in by_query.values():
    rejected = [i for i, a in enumerate(steps) if not a['accepted']]
    if not rejected:
        continue
    rejected_queries += 1
    never_recovered += all(not a['accepted'] for a in steps[rejected[0]:])
    seen = []
    for i in rejected:
        repeat_rejections += steps[i]['candidate_plan'] in seen
        seen.append(steps[i]['candidate_plan'])
n_rejected = sum(not a['accepted'] for a in attempts)

# Tokens and time spent on rejected candidates.
def total(key, rows):
    return sum(a.get(key) or 0 for a in rows)
rejected_rows = [a for a in attempts if not a['accepted']]
tok_rej, tok_all = total('total_tokens_delta', rejected_rows), total('total_tokens_delta', attempts)
sec_rej, sec_all = total('attempt_time_seconds', rejected_rows), total('attempt_time_seconds', attempts)

# Commonsense fixes in rejected candidates, replayed against the plan R2 kept.
cs_discarded = 0
for path in result_files:
    with open(path, encoding='utf-8') as f:
        result = json.load(f)
    history = {h['step']: h for h in result['repair_history']}
    kept = Counter(result['initial_verification']['commonsense_errors'])
    for a in result['repair_attempts']:
        candidate = Counter(history[a['step']]['verification']['commonsense_errors'])
        if a['accepted']:
            kept = candidate
        else:
            cs_discarded += sum((kept - candidate).values())
    assert kept == Counter(result['final_verification']['commonsense_errors']), path

def share(part, whole):
    return fmt_pct(part / whole * 100)

followup = [
    ('Queries never accepted again after first rejection',
     f'{never_recovered} / {rejected_queries} ({share(never_recovered, rejected_queries)}%)'),
    ('Rejected candidates identical to an earlier rejected one',
     f'{repeat_rejections} / {n_rejected} ({share(repeat_rejections, n_rejected)}%)'),
    ('Tokens spent on rejected candidates',
     f'{tok_rej / 1e6:.2f}M / {tok_all / 1e6:.2f}M ({share(tok_rej, tok_all)}%)'),
    ('Time spent on rejected candidates',
     f'{sec_rej:,.0f} s / {sec_all:,.0f} s ({share(sec_rej, sec_all)}%)'),
    ('Commonsense fixes discarded by rejections', cs_discarded),
]
specific += followup
for k, v in followup:
    print(f'  {k:60} {v}')

  Queries never accepted again after first rejection           5 / 5 (100%)
  Rejected candidates identical to an earlier rejected one     6 / 25 (24%)
  Tokens spent on rejected candidates                          0.89M / 5.63M (15.72%)
  Time spent on rejected candidates                            803 s / 6,261 s (12.82%)
  Commonsense fixes discarded by rejections                    64


## Step 4 — Assemble the full table
The summary sentence is built from the computed values, so it stays in sync with the table.

In [8]:
rows = [('Metric', f'{STRATEGY} training result')]
rows += [(k, v) for k, v in core]
rows += [('', '')]  # spacer between core and strategy-specific
rows += [(k, v) for k, v in specific]

harmful = int(float(mech['harmful_candidate_blocked']['count']))
harmful_pct = float(mech['harmful_candidate_blocked']['rate']) * 100
summary = (f'R2 rejected {harmful} harmful candidates ({harmful_pct:.1f}% of attempts), '
           'preventing any protected-constraint regression. This is consistent with its design. '
           f'The cost is caution: only {personal_fixes} personal fixes and '
           f'~{round(unchanged_pct)}% of attempts unchanged.')

for k, v in rows:
    print(f'{k:60} | {v}')
print()
print('Summary:', summary)

Metric                                                       | R2 training result
Queries                                                      | 60
Repair attempts                                              | 332
Personal repair attempts                                     | 141
Commonsense repair attempts                                  | 191
Confirmed unchanged attempts                                 | 235 (70.78%)
Personal constraint fixes                                    | 2
Personal regressions                                         | 0
                                                             | 
Harmful candidates blocked (rejected)                        | 25 / 332 (7.53%)
Rejected: fixed target but caused a protected regression     | 0 / 332 (0%)
Rejected and loop continued                                  | 20 / 332 (6.02%)
Rejected candidate differed from accepted plan               | 25 / 25 (100%)
Commonsense errors (start → end)                             | 304 

## Step 5 — Write the exact table to `Train_results_R2.xlsx`
Laid out like the R2 block in `Results.xlsx`: a title row, the bordered table, and the summary in a merged cell underneath.

In [9]:
import openpyxl
from openpyxl.styles import Alignment, Border, Font, Side
from openpyxl.cell.rich_text import CellRichText, TextBlock
from openpyxl.cell.text import InlineFont

wb = openpyxl.Workbook()
ws = wb.active
ws.title = f'{STRATEGY} train results'
thin = Side(style='thin')
box = Border(left=thin, right=thin, top=thin, bottom=thin)

# Title row, then the table starting on row 2.
ws.cell(1, 1, STRATEGY)
TOP = 2
for r, (k, v) in enumerate(rows, start=TOP):
    ws.cell(r, 1, k).border = box
    value = ws.cell(r, 2, v)
    value.border = box
    value.alignment = Alignment(horizontal='right')

# Bold, centred header row.
for col in (1, 2):
    ws.cell(TOP, col).font = Font(bold=True)
    ws.cell(TOP, col).alignment = Alignment(horizontal='center')

# Summary in a merged, wrapped, bordered block under the table.
summary_row = TOP + len(rows) + 1
ws.merge_cells(start_row=summary_row, start_column=1, end_row=summary_row, end_column=2)
cell = ws.cell(summary_row, 1)
cell.value = CellRichText(TextBlock(InlineFont(b=True), 'Summary'), ': ' + summary)
cell.alignment = Alignment(wrap_text=True, vertical='bottom')
for col in (1, 2):
    ws.cell(summary_row, col).border = box
ws.row_dimensions[summary_row].height = 60

ws.column_dimensions['A'].width = 52
ws.column_dimensions['B'].width = 40

out_xlsx = NB_DIR / f'Train_results_{STRATEGY}.xlsx'
wb.save(out_xlsx)
print('Wrote', out_xlsx)
print('Rows written:', len(rows), '+ title + summary')

Wrote C:\Users\Xue Jiarui\OneDrive - National University of Singapore\Documents\Agentic-Travel-Planner-Ablation-of-Verifications-\train_result_analysis\Train_results_R2.xlsx
Rows written: 19 + title + summary


## Step 6 — Read the file back to confirm it saved correctly

In [10]:
chk = openpyxl.load_workbook(out_xlsx, data_only=True, rich_text=True)
cs = chk.active
print('Sheet:', cs.title, '| dims:', cs.dimensions)
print()
for row in cs.iter_rows(values_only=True):
    if any(c not in (None, '') for c in row):
        print(f'{str(row[0]):52} | {row[1] if row[1] is not None else ""}')

Sheet: R2 train results | dims: A1:B22

R2                                                   | 
Metric                                               | R2 training result
Queries                                              | 60
Repair attempts                                      | 332
Personal repair attempts                             | 141
Commonsense repair attempts                          | 191
Confirmed unchanged attempts                         | 235 (70.78%)
Personal constraint fixes                            | 2
Personal regressions                                 | 0
Harmful candidates blocked (rejected)                | 25 / 332 (7.53%)
Rejected: fixed target but caused a protected regression | 0 / 332 (0%)
Rejected and loop continued                          | 20 / 332 (6.02%)
Rejected candidate differed from accepted plan       | 25 / 25 (100%)
Commonsense errors (start → end)                     | 304 → 271 (−10.9%)
Queries never accepted again after first rejection   